# Check the saved IOI results

In a prompt like “When Paul and George went to the store, Paul gave a bottle to”, GPT-2 gives each possible next name a score. This notebook uses scores and model values saved from the original run. You can check the results on a CPU. Start Jupyter in the project folder with NumPy, scikit-learn, and Matplotlib installed.

In [ ]:
import os
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "analysis" / "verify.py").is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from analysis.verify import REFERENCE, RUN, compare, load_json, recompute
metrics = recompute()
compare(metrics, load_json(REFERENCE))
print("Checks passed. Selected feature:", metrics["feature_id"])
print("Split sizes:", metrics["split_sizes"])

## Rebuild the values and predict name order

The SAE tries to rebuild the values from GPT-2. Explained variance tells us how well it does. A negative value means that guessing the average for each dimension worked better. AUROC tells us how well the feature predicts whether the recipient came first. A score of 0.5 is chance and 1 is perfect.

In [ ]:
for name in ("dev", "test", "ood"):
    m = metrics["splits"][name]
    print(f"{name:5s}  explained variance {m['reconstruction']['explained_variance']:8.2%}  "
          f"order AUROC {m['order_auroc']:.4f}  active prompts {m['active_prompts']}")

## Remove the feature and compare the two names

The score gap is the recipient's logit minus the subject's logit. We read the original gap and the change for each prompt from a CSV file. A negative change means the gap got smaller. Scores in the file have five decimal places.

In [ ]:
import csv
for split in ("test", "ood"):
    v = metrics["splits"][split]["logits"]
    ci = v["mean_change_ci95"]
    print(f"{split:5s}  mean {v['mean_change']:+.5f}  "
          f"95% interval [{ci[0]:+.5f}, {ci[1]:+.5f}]  "
          f"prompts switching names {v['pairwise_decisions_changed']}")
with (RUN / "ood_examples.csv").open(newline="") as f:
    examples = list(csv.DictReader(f))
example = next(r for r in examples if float(r["selected_feature_ablation_gap_change"]) < 0)
print(example["prompt"])
print("Correct name:", example["correct_io"], "| Subject:", example["subject"])
print("Gap change:", example["selected_feature_ablation_gap_change"])

## Make the two figures again

In the grid, each cell shows how many gaps changed out of all prompts in that group. For example, 19/21 means 19 of the 21 gaps changed. The last two rows use the new sentence patterns.

In [ ]:
import tempfile
import matplotlib.pyplot as plt
from analysis.figures import draw_figures

with tempfile.TemporaryDirectory() as temp_dir:
    directory = Path(temp_dir)
    draw_figures(metrics, directory)
    for name in ("reconstruction_and_order.png", "ablation_by_template.png"):
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.imshow(plt.imread(directory / name))
        ax.axis("off")
        fig.tight_layout()
        plt.show()
        plt.close(fig)

The [research report](../reports/research_report.md) explains why the sentence pattern matters and how we checked the results.